**Install Libraries**

In [ ]:
# Section 1 : Install Required Libraries

!pip -q install \
langchain==0.3.27 \
langchain-community==0.3.27 \
langchain-google-genai==2.1.12 \
langchain-huggingface==0.3.1 \
langchain-text-splitters==0.3.9 \
sentence-transformers==5.1.0 \
faiss-cpu==1.12.0 \
pypdf==5.9.0

**Import Libraries**

In [ ]:
# Section 2 : Import Libraries

import os
import warnings

warnings.filterwarnings("ignore")

from google.colab import userdata
from google.colab import files

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

print("✅ All Libraries Imported Successfully")

**Checking versions**

In [ ]:
import langchain
import sentence_transformers

print("LangChain :", langchain.__version__)
print("Sentence Transformers :", sentence_transformers.__version__)

**Upload PDF**

In [ ]:
# Section 3 : Upload PDF

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("✅ Uploaded File :", pdf_file)

**Read PDF**

In [ ]:
# Section 4 : Read PDF

reader = PdfReader(pdf_file)

print("Total Pages :", len(reader.pages))

**Extract text**

In [ ]:
# Section 5 : Extract Text

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("✅ Text Extracted Successfully")
print("\nTotal Characters :", len(text))

**Preview extracted text**

In [ ]:
# Section 6 : Preview Text

print(text[:1500])

**Split texts into chunks**

In [ ]:
# Section 7 : Split Text

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)

print("✅ Total Chunks :", len(chunks))

**Preview chunks**

In [ ]:
# Section 8 : Preview Chunks

print(chunks[0])

**Create Embedding Model**

In [ ]:
# Section 9 : Load Embedding Model

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Embedding Model Loaded Successfully")

**Generate Vector Database (FAISS)**

In [ ]:
# Section 10 : Create FAISS Vector Database

vector_db = FAISS.from_texts(
    texts=chunks,
    embedding=embedding_model
)

print("✅ FAISS Vector Database Created Successfully")

**Save vector database**

In [ ]:
# Section 11 : Save FAISS Database

vector_db.save_local("faiss_index")

print("✅ Vector Database Saved Successfully")

In [ ]:
# Section 12 : Similarity Search Test

query = "What is machine learning?"

results = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print("=" * 70)
    print(f"Result {i}")
    print("=" * 70)
    print(doc.page_content[:500])
    print()

In [ ]:
# Section 13 : Load Gemini API Key

from google.colab import userdata
import os

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("✅ Gemini API Key Loaded Successfully")

In [ ]:
# Section 14 : Initialize Gemini

from langchain_google_genai import ChatGoogleGenerativeAI

# Try one of these models if another gives a 404
MODEL_NAME = "gemini-2.0-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0
)

print(f"✅ Gemini Initialized Successfully")
print(f"Using Model: {MODEL_NAME}")

**Create the Retriever**

In [ ]:
# Section 15 : Create Retriever

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("✅ Retriever Created Successfully")

**Asking questions**

In [ ]:
# Section 16 : Ask Questions

question = input("Enter your question: ")

docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

print("✅ Relevant Context Retrieved")

**Generate answer with gemini**

In [ ]:
# Section 17 : Generate Answer

import time

start = time.time()

# Retrieve relevant documents
docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are EduGenie, an intelligent AI tutor.

Answer ONLY using the provided context.

Rules:
1. Do not make up answers.
2. If the answer is not present in the context, reply:
"I couldn't find that information in the uploaded PDF."

Context:
{context}

Question:
{question}

Answer:
"""

try:
    response = llm.invoke(prompt)

    print("\n" + "="*70)
    print("📚 EduGenie Answer")
    print("="*70)
    print(response.content)

except Exception as e:
    print("❌ Error while generating response:")
    print(e)

end = time.time()

print("\n" + "="*70)
print(f"⏱️ Response Time: {end-start:.2f} seconds")
print("="*70)